# Weighted Ensemble: XGBoost + LightGBM

This notebook trains and compares individual XGBoost and LightGBM models with Optuna-tuned hyperparameters, then combines them via soft-voting ensemble to achieve potentially better test performance.

## Approach
1. Load and preprocess the emotion dataset
2. Train both XGBoost and LightGBM using the best parameters from `advanced_ml_models.ipynb`
3. Create a weighted ensemble using validation F1 scores as weights
4. Compare single-model vs ensemble metrics with confusion matrices and class breakdowns

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import classification_report, cohen_kappa_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
RANDOM_STATE = 42

FEATURE_COLS = [
    "F0_mean", "F0_std", "F0_range",
    "Energy_ mean", "Energy_ std",
    "ZCR_mean", "ZCR_std",
    "Spectral_centroid_mean", "Spectral_centroid_std", "Spectral_flux_mean",
    "MFCC_C0_mean", "MFCC_C1_mean", "MFCC_C2_mean", "MFCC_C3_mean",
    "MFCC_C5_mean", "MFCC_C7_mean", "MFCC_C10_mean",
    "MFCC_C0_std", "MFCC_C1_std", "MFCC_C2_std", "MFCC_C3_std",
    "MFCC_C5_std", "MFCC_C7_std",
    "Delta_MFCC_C0_std", "Delta_MFCC_C2_std", "Delta_MFCC_C3_std",
]

# Best parameters from Optuna tuning in advanced_ml_models.ipynb
XGB_PARAMS = {
    'n_estimators': 468,
    'max_depth': 10,
    'learning_rate': 0.17549140891728818,
    'subsample': 0.9690394070981359,
    'colsample_bytree': 0.7725519831966194,
    'gamma': 1.0492767129301485e-08,
}

LGB_PARAMS = {
    'n_estimators': 499,
    'max_depth': 11,
    'num_leaves': 67,
    'learning_rate': 0.24625126683753454,
    'subsample': 0.6909971314141472,
    'colsample_bytree': 0.7554088091076056,
}

# Validation F1 scores to weight the ensemble
XGB_VAL_F1 = 0.8334582426563992
LGB_VAL_F1 = 0.842069322599305

### Load & Preprocess Data

In [ ]:
# Resolve CSV path (prefer dataset/ but fall back to root)
_base = os.getcwd()
_project_root = os.path.dirname(_base) if os.path.basename(_base).lower() == "training" else _base
ALL_EMOTIONS_CSV = os.path.normpath(os.path.join(_project_root, "dataset", "all_emotions.csv"))

if not os.path.isfile(ALL_EMOTIONS_CSV):
    fallback_csv = os.path.normpath(os.path.join(_project_root, "all_emotions.csv"))
    if os.path.isfile(fallback_csv):
        ALL_EMOTIONS_CSV = fallback_csv

print(f"Loading from: {ALL_EMOTIONS_CSV}")
df = pd.read_csv(ALL_EMOTIONS_CSV)
print(f"Raw shape: {df.shape}")

In [ ]:
TARGET_COL = "label"
if TARGET_COL not in df.columns and "Label" in df.columns:
    TARGET_COL = "Label"

# Clean null targets and 'nan' strings
df_cleaned = df.dropna(subset=[TARGET_COL]).copy()
df_cleaned = df_cleaned[df_cleaned[TARGET_COL].astype(str).str.strip().str.lower() != "nan"]

# Impute missing values in features
for col in FEATURE_COLS:
    s = pd.to_numeric(df_cleaned[col], errors="coerce")
    s = s.replace([np.inf, -np.inf], np.nan)
    med = s.median()
    if pd.isna(med):
        med = 0.0
    df_cleaned[col] = s.fillna(med)

X = df_cleaned[FEATURE_COLS].copy()
y = df_cleaned[TARGET_COL].astype(str).str.strip()

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Preprocessed X shape: {X.shape}")
print(f"Classes: {le.classes_}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train shape: {X_train_scaled.shape} | Test shape: {X_test_scaled.shape}")

### Train Models

In [ ]:
xgb_model = xgb.XGBClassifier(
    **XGB_PARAMS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric='mlogloss',
    objective='multi:softprob',
    num_class=len(le.classes_),
)

lgb_model = lgb.LGBMClassifier(
    **LGB_PARAMS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
    objective='multiclass',
    num_class=len(le.classes_),
)

print("Training XGBoost...")
xgb_model.fit(X_train_scaled, y_train)
print("Training LightGBM...")
lgb_model.fit(X_train_scaled, y_train)
print("Done!")

### Individual Model Evaluation

In [ ]:
xgb_pred = xgb_model.predict(X_test_scaled)
lgb_pred = lgb_model.predict(X_test_scaled)

xgb_f1 = f1_score(y_test, xgb_pred, average="weighted")
xgb_kappa = cohen_kappa_score(y_test, xgb_pred)

lgb_f1 = f1_score(y_test, lgb_pred, average="weighted")
lgb_kappa = cohen_kappa_score(y_test, lgb_pred)

print("\n=== XGBoost ===")
print(classification_report(y_test, xgb_pred, target_names=le.classes_))
print(f"XGBoost weighted F1: {xgb_f1:.4f}")
print(f"XGBoost Cohen Kappa: {xgb_kappa:.4f}")

print("\n=== LightGBM ===")
print(classification_report(y_test, lgb_pred, target_names=le.classes_))
print(f"LightGBM weighted F1: {lgb_f1:.4f}")
print(f"LightGBM Cohen Kappa: {lgb_kappa:.4f}")

### Weighted Soft-Voting Ensemble

In [ ]:
# Get probability predictions
xgb_proba = xgb_model.predict_proba(X_test_scaled)
lgb_proba = lgb_model.predict_proba(X_test_scaled)

# Weighted average of probabilities (using validation F1 as weights)
ensemble_weights = np.array([XGB_VAL_F1, LGB_VAL_F1], dtype=float)
ensemble_proba = (
    ensemble_weights[0] * xgb_proba + ensemble_weights[1] * lgb_proba
) / ensemble_weights.sum()
ensemble_pred = np.argmax(ensemble_proba, axis=1)

ensemble_f1 = f1_score(y_test, ensemble_pred, average="weighted")
ensemble_kappa = cohen_kappa_score(y_test, ensemble_pred)
ensemble_report = classification_report(y_test, ensemble_pred, target_names=le.classes_, output_dict=True)

print("\n=== Weighted Ensemble (XGBoost + LightGBM) ===")
print(classification_report(y_test, ensemble_pred, target_names=le.classes_))
print(f"Ensemble weighted F1: {ensemble_f1:.4f}")
print(f"Ensemble Cohen Kappa: {ensemble_kappa:.4f}")

### Research Visualizations

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (pred, title) in enumerate([
    (xgb_pred, "XGBoost"),
    (lgb_pred, "LightGBM"),
    (ensemble_pred, "Ensemble"),
]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx], cbar=False)
    axes[idx].set_title(f"{title} Confusion Matrix")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("True")

plt.tight_layout()
plt.show()

In [ ]:
# Model comparison
models = ["XGBoost", "LightGBM", "Ensemble"]
f1_scores = [xgb_f1, lgb_f1, ensemble_f1]
kappas = [xgb_kappa, lgb_kappa, ensemble_kappa]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - width / 2, f1_scores, width, label="Weighted F1", color="skyblue")
plt.bar(x + width / 2, kappas, width, label="Cohen Kappa", color="steelblue")
plt.xticks(x, models)
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Model Comparison: F1 vs Cohen Kappa")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Per-class F1 for ensemble
class_f1 = [ensemble_report[label]["f1-score"] for label in le.classes_]

plt.figure(figsize=(8, 5))
sns.barplot(x=list(le.classes_), y=class_f1, color="steelblue")
plt.ylim(0, 1)
plt.ylabel("F1-score")
plt.xlabel("Emotion Class")
plt.title("Ensemble Per-Class F1-Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
print("\n=== Summary ===")
print(f"XGBoost: F1={xgb_f1:.4f}, Kappa={xgb_kappa:.4f}")
print(f"LightGBM: F1={lgb_f1:.4f}, Kappa={lgb_kappa:.4f}")
print(f"Ensemble: F1={ensemble_f1:.4f}, Kappa={ensemble_kappa:.4f}")
